# Model Monitoring v2 on Vertex AI 

## Overview

### What is Model Monitoring?

Model monitoring is only one piece of the ML Ops puzzle - it helps answer the following questions:

* How well do recent service requests match the training data used to build your model? This is called **training-serving skew**.
* How significantly are service requests evolving over time? This is called **drift detection**.

If production traffic differs from  training data, or varies substantially over time, that's likely to impact the quality of the answers your model produces. When that happens, you'd like to be alerted automatically and responsively.

Vertex AI Model Monitoring provides two offerings: v2 and v1.

Model Monitoring v2 is in Preview and is the latest offering that associates all monitoring tasks with a model version. In contrast, Model Monitoring v1 is Generally Available and is configured on Vertex AI endpoints.

This tutorial demonstrates how to use the Vertex AI SDK for Python to set up Vertex AI Model Monitoring V2 for your model.
Model Monitoring v2 lets you track metrics over time after you configure a model monitor and run monitoring jobs. You can run on-demand monitoring jobs or set up scheduled runs. By using scheduled runs, Model Monitoring automatically runs monitoring jobs based on a schedule that you define.

## Learning Objectives
In this tutorial, you'll complete the following steps:

- Upload a custom model to Vertex AI Model Registry.
- Deploy the model to Vertex AI Endpoint with Request-Response Logging enabled.
- Generate some online prediction traffic.
- Create a model monitor.
- Run an on-demand model monitoring job to analyze data drift between the Online prediction traffic and the training dataset.
- Run an on-demand model monitoring job to analyze feature attribution drift between the Online prediction traffic and a GCS baseline dataset.
- Create a schedule to continuously run model monitoring jobs to analyze data drift between the Online prediction traffic and the training dataset.

## Getting Started

### Set Google Cloud project information nad environment variables

In [ ]:
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
BUCKET = f"{PROJECT}-monitoring"
REGION = "us-central1"
SUFFIX = "aiplatform.googleapis.com"
API_ENDPOINT = f"{REGION}-{SUFFIX}"
PREDICT_API_ENDPOINT = f"{REGION}-prediction-{SUFFIX}"

!gcloud config set project $PROJECT
!gcloud config set ai/region $REGION

%env PROJECT={PROJECT}
%env BUCKET={BUCKET}
%env REGION={REGION}
%env GOOGLE_CLOUD_PROJECT={PROJECT}

%load_ext google.cloud.bigquery

### Import required your dependencies

In [ ]:
import os
import vertexai
import google.cloud.aiplatform as aiplatform
import pandas as pd
import pprint as pp
import sys
import time
import pandas as pd
from google.protobuf import timestamp_pb2
from google.cloud import bigquery
from vertexai.resources.preview import ml_monitoring
from google.cloud.aiplatform_v1beta1.types import (ExplanationMetadata,
                                                   ExplanationParameters,
                                                   ExplanationSpec)

### initialize Vertex AI SDK

In [ ]:
vertexai.init(project=PROJECT, location=REGION)

## Start Model Monitoring tutorial

### Step 1: Create a Cloud Storage bucket

Create a Cloud Storage bucket to store intermediate artifacts such as datasets.

In [ ]:
BUCKET_URI = f"gs://model-monitoring-{PROJECT}-v2"
%env BUCKET_URI={BUCKET_URI}

In [ ]:
%%bash
if gsutil ls -b "${BUCKET_URI}" >/dev/null 2>&1; then
   echo -e "Bucket already exists."
else
   echo "Creating a new GCS bucket."
   gsutil mb -p "${PROJECT}" -l "${REGION}" "${BUCKET_URI}"
   echo -e "\nHere are your current buckets:"
   gsutil ls
fi

### Step 2: Prepare a model in Vertex AI Model Registry

### The example model

The model you'll use in this notebook is based on [this blog post](https://cloud.google.com/blog/topics/developers-practitioners/churn-prediction-game-developers-using-google-analytics-4-ga4-and-bigquery-ml). The idea behind this model is that your company has extensive log data describing how your game users have interacted with the site. The raw data contains the following categories of information:

- identity - unique player identity numbers
- demographic features - information about the player, such as the geographic region in which a player is located
- behavioral features - counts of the number of times a  player has triggered certain game events, such as reaching a new level
- churn propensity - this is the label or target feature, it provides an estimated probability that this player will churn, i.e. stop being an active player.

The blog article referenced above explains how to use BigQuery to store the raw data, pre-process it for use in machine learning, and train a model.

Register a model in Vertex AI Model Registry with its artifacts, enabling you to perform online serving or batch prediction.

In [ ]:
MODEL_PATH = "gs://cloud-samples-data/vertex-ai/model-deployment/models/churn"
MODEL_NAME = "churn"
SERVING_CONTAINER = (
    "us-docker.pkg.dev/cloud-aiplatform/prediction/tf2-cpu.2-12:latest"
)

model = aiplatform.Model.upload(
    display_name=MODEL_NAME,
    artifact_uri=MODEL_PATH,
    serving_container_image_uri=SERVING_CONTAINER,
    sync=True,
)

MODEL_ID = model.resource_name.split("/")[-1]

### Step 3: Deploy Model to Vertex AI Endpoint with Request-Respponse Logging Enabled

#### Create an endpoint with logging enabled and deploy the model to this endpoint.
Specify the logging sampling rate and BigQuery destination.

In [ ]:
TIMESTAMP = pd.Timestamp.utcnow().strftime("%Y%m%d%H%M%S")
ENDPOINT_DISPLAY_NAME = f"churn_endpoint_{TIMESTAMP}"
BQ_LOGGING_DATASET = f"churn_production"
BQ_LOGGING_TABLE = f"bq://{PROJECT}.{BQ_LOGGING_DATASET}.req_resp"

endpoint = aiplatform.Endpoint.create(
    display_name=ENDPOINT_DISPLAY_NAME,
    enable_request_response_logging=True,
    request_response_logging_sampling_rate=1.0,
    request_response_logging_bq_destination_table=BQ_LOGGING_TABLE,
)

# Deploy model to this endpoint
endpoint.deploy(
    model=model,
    traffic_percentage=100,
    min_replica_count=1,
    max_replica_count=1,
    machine_type="n1-standard-4",
)

## Run a prediction test

Now that you have imported a model and deployed that model to an endpoint, you are ready to verify that it's working. Run the next cell to send a test prediction request. If everything works as expected, you should receive a JSON response.

In [ ]:
DEFAULT_INPUT = {
    "cnt_ad_reward": 0,
    "cnt_challenge_a_friend": 0,
    "cnt_completed_5_levels": 1,
    "cnt_level_complete_quickplay": 3,
    "cnt_level_end_quickplay": 5,
    "cnt_level_reset_quickplay": 2,
    "cnt_level_start_quickplay": 6,
    "cnt_post_score": 34,
    "cnt_spend_virtual_currency": 0,
    "cnt_use_extra_steps": 0,
    "cnt_user_engagement": 120,
    "country": "Denmark",
    "dayofweek": 3,
    "julianday": 254,
    "language": "da-dk",
    "month": 9,
    "operating_system": "IOS",
    "user_pseudo_id": "104B0770BAE16E8B53DF330C95881893",
}

In [ ]:
try:
    resp = endpoint.predict([DEFAULT_INPUT])
    for i in resp.predictions:
        vals = i["churned_values"]
        probs = i["churned_probs"]
    for i in range(len(vals)):
        print(vals[i], probs[i])
    pp.pprint(resp)
except Exception as ex:
    print("prediction request failed", ex)

Taking a closer look at the results, we see the following elements:

- **churned_values** - a set of possible values (0 and 1) for the target field
- **churned_probs** - a corresponding set of probabilities for each possible target field value 
- **predicted_churn** - based on the churned_probs, (1) has a higher probability so the predicted value of the target field is (1)

### Create BigQuery Table with training data
To compute the skew between training data and live data, we need to provide Vertex AI with the training data. The training data is hosted as a CSV file in a public GCS bucket. The following cells create a BigQuery dataset, and ingest the training data into a BigQuery table. You will provide this BigQuery table to Vertex in the Monitoring Job configuration, enabling it to compute skew detection.

In [ ]:
%%bash
DATASET_LOCATION=US
DATASET_ID=churn_dataset
TABLE_ID=train
DATA_SOURCE=gs://asl-public-data/data/churn/train.csv

bq --location=$DATASET_LOCATION --project_id=$PROJECT mk --dataset $DATASET_ID

bq --project_id=$PROJECT --dataset_id=$DATASET_ID load \
--source_format=CSV \
--autodetect \
--replace \
$TABLE_ID \
$DATA_SOURCE \

Verify the dataset and table were succesfully created in your project, and the training data is there

In [ ]:
%%bigquery
SELECT * 
FROM churn_dataset.train
LIMIT 10

#### Send some online prediction requests to the endpoint
Here we just use subset of the training dataset as prediction data

In [ ]:
DATASET_BQ_URI = f"bq://{PROJECT}.churn_dataset.train"
GROUND_TRUTH = "churned"

bqclient = bigquery.Client(project=PROJECT)
table = bigquery.TableReference.from_string(DATASET_BQ_URI[5:])

rows = bqclient.list_rows(table, max_results=1000)
count = 0
for row in rows:
    instance = {}
    for key, value in row.items():
        if key == GROUND_TRUTH:
            continue
        if value is None:
            value = ""
        instance[key] = value
    endpoint.predict(instances=[instance])
    # Print progress
    if count % 100 == 0:
        print(f"Sent: {count} requests")
    count = count + 1
    time.sleep(0.01)

#### Check the endpoint logging table.

In [ ]:
%%bigquery
SELECT * FROM `churn_production.req_resp` ORDER BY logging_time DESC LIMIT 10

### Step 4: Create a Model Monitoring

#### Define Model Monitoring Schema

The monitoring schema is a required configuration for a model monitor. The schema consists of input features names, prediction outputs, and ground truth (if available), along with their respective data types.

**Note: The schema is optional only for AutoML tables (Regression/Classification), as it will be automatically fetched when available (if Model Monitoring is unable to retrieve the schema, you will need to provide your own).**

In [ ]:
MODEL_MONITORING_SCHEMA = ml_monitoring.spec.ModelMonitoringSchema(
    feature_fields=[
        ml_monitoring.spec.FieldSchema(name="user_pseudo_id", data_type="string"),
        ml_monitoring.spec.FieldSchema(name="country", data_type="string"),
        ml_monitoring.spec.FieldSchema(name="operating_system", data_type="string"),
        ml_monitoring.spec.FieldSchema(name="cnt_user_engagement", data_type="integer"),
        ml_monitoring.spec.FieldSchema(
            name="cnt_level_start_quickplay", data_type="integer"
        ),
        ml_monitoring.spec.FieldSchema(
            name="cnt_level_end_quickplay", data_type="integer"
        ),
        ml_monitoring.spec.FieldSchema(
            name="cnt_level_complete_quickplay", data_type="integer"
        ),
        ml_monitoring.spec.FieldSchema(
            name="cnt_level_reset_quickplay", data_type="integer"
        ),
        ml_monitoring.spec.FieldSchema(name="cnt_post_score", data_type="integer"),
        ml_monitoring.spec.FieldSchema(
            name="cnt_spend_virtual_currency", data_type="integer"
        ),
        ml_monitoring.spec.FieldSchema(name="cnt_ad_reward", data_type="integer"),
        ml_monitoring.spec.FieldSchema(
            name="cnt_challenge_a_friend", data_type="integer"
        ),
        ml_monitoring.spec.FieldSchema(
            name="cnt_completed_5_levels", data_type="integer"
        ),
        ml_monitoring.spec.FieldSchema(name="cnt_use_extra_steps", data_type="integer"),
        ml_monitoring.spec.FieldSchema(name="month", data_type="categorical"),
        ml_monitoring.spec.FieldSchema(name="julianday", data_type="integer"),
        ml_monitoring.spec.FieldSchema(name="dayofweek", data_type="integer"),
    ],
    ground_truth_fields=[
        ml_monitoring.spec.FieldSchema(name="churned", data_type="categorical")
    ],
    prediction_fields=[
        ml_monitoring.spec.FieldSchema(
            name="predicted_churned", data_type="categorical"
        )
    ],
)

The training dataset can serve as the baseline dataset to analysis the data drift in production. You can register the training dataset in the model monitor.

In [ ]:
PUBLIC_TRAINING_DATASET = "gs://asl-public-data/data/churn/train.csv"
TRAINING_URI = f"{BUCKET_URI}/model-monitoring/churn/churn_training.csv"

#### Copy files to your projects gs bucket to avoid permission issues.
Ignore any error(s) for bucket already exists.

In [ ]:
!gcloud storage cp $PUBLIC_TRAINING_DATASET $TRAINING_URI

In [ ]:
TRAINING_DATASET = ml_monitoring.spec.MonitoringInput(
    gcs_uri=TRAINING_URI, data_format="csv"
)

#### Create a model monitor

A model monitor is a top-level resource to manage your metrics and model monitoring jobs.

In [ ]:
my_model_monitor = ml_monitoring.ModelMonitor.create(
    project=PROJECT,
    location=REGION,
    display_name="churn_model_monitor",
    model_name=model.resource_name,
    model_version_id="1",
    training_dataset=TRAINING_DATASET,
    model_monitoring_schema=MODEL_MONITORING_SCHEMA,
)
MODEL_MONITOR_ID = my_model_monitor.name
print(f"MODEL MONITOR {MODEL_MONITOR_ID} created.")

### Step 5: Run an on-demand model monitoring job

#### Define the monitoring objective configs

For tabular models, Model Monitoring supports the following objectives:

*   **Input feature drift detection**

    Model Monitoring offers drift analysis for both categorical and numeric feature types, with the following supported metrics:

    *    Categorical Feature: `Jensen Shannon Divergence`, `L Infinity`
    *    Numeric Feature: `Jensen Shannon Divergence`

    You can choose to analyze only the features of interest by specifying them in the `features` fields of the `ml_monitoring.spec.DataDriftSpec` specification. If not specified, all input features in the model schema are analyzed. Additionally, you have the option to set default thresholds for categorical or numeric features, or you can specify thresholds for individual features. If the detected drift surpasses a threshold, an alert is sent through email or another notification channel.

*  **Prediction output drift detection**

    Similar to input feature drift detection, prediction output drift detection identifies data drift in the prediction outputs.

*   **Feature attribution drift detection**

    Model Monitoring leverages Vertex Explainable AI to monitor feature attributions. Explainable AI enables you to understand the relative contribution of each feature to a resulting prediction. In essence, it assesses the magnitude of each feature's influence.
    You must configure the `Explanation` specification with the feature attribution objectives configuration.


Input feature drift specification

In [ ]:
FEATURE_THRESHOLDS = {
    "country": 0.003,
    "cnt_user_engagement": 0.004,
}

FEATURE_DRIFT_SPEC = ml_monitoring.spec.DataDriftSpec(
    categorical_metric_type="l_infinity",
    numeric_metric_type="jensen_shannon_divergence",
    default_categorical_alert_threshold=0.2,
    default_numeric_alert_threshold=0.3,
    feature_alert_thresholds=FEATURE_THRESHOLDS,
)

Prediction output drift specification

In [ ]:
PREDICTION_OUTPUT_DRIFT_SPEC = ml_monitoring.spec.DataDriftSpec(
    categorical_metric_type="l_infinity",
    numeric_metric_type="jensen_shannon_divergence",
    default_categorical_alert_threshold=0.1,
    default_numeric_alert_threshold=0.1,
)

Feature attribution specification

In [ ]:
FEATURE_ATTRIBUTION_SPEC = ml_monitoring.spec.FeatureAttributionSpec(
    default_alert_threshold=0.0003,
    feature_alert_thresholds={"cnt_ad_reward": 0.0001},
)

#### Define the alert notification and metrics output spec.

Model Monitoring supports the following notification methods:

*   Email
*   [Notification Channel](https://cloud.google.com/monitoring/support/notification-options)
*   [Cloud Logging](https://cloud.google.com/logging/docs)  

This notebook uses email as an example.

Export generated metrics to the Google Cloud Storage location that you specified or, if you don't specify a location, Vertex AI creates a default bucket to use.

In [ ]:
EMAIL = "[PUT_YOUR@EMAIL.HERE]" #TODO: ADD YOUR EMAIL HERE

NOTIFICATION_SPEC = ml_monitoring.spec.NotificationSpec(
    user_emails=[EMAIL],
)

OUTPUT_SPEC = ml_monitoring.spec.OutputSpec(gcs_base_dir=BUCKET_URI)

#### Run Model Monitoring Jobs

##### **Example 1: Feature drift & Prediction output drift detection, compares current data with the training dataset.**

In [ ]:
TIMESTAMP = pd.Timestamp.utcnow().strftime("%Y%m%d%H%M%S")
JOB_DISPLAY_NAME = f"churn_model_monitoring_job_{TIMESTAMP}"
TARGET_DATASET = ml_monitoring.spec.MonitoringInput(endpoints=[endpoint.resource_name])
model_monitoring_job_1 = my_model_monitor.run(
    display_name=JOB_DISPLAY_NAME,
    baseline_dataset=TRAINING_DATASET,
    target_dataset=TARGET_DATASET,
    tabular_objective_spec=ml_monitoring.spec.TabularObjective(
        # Input feature drift spec.
        feature_drift_spec=FEATURE_DRIFT_SPEC,
        # Prediction output drift spec.
        prediction_output_drift_spec=PREDICTION_OUTPUT_DRIFT_SPEC,
    ),
    notification_spec=NOTIFICATION_SPEC,
    output_spec=OUTPUT_SPEC,
)

##### **Example 2: Feature attribution drift detection, compares current data with a Cloud Storage baseline dataset**

For feature attribution monitoring, the dataset is sent to the Vertex AI batch explanation job in the following way:

*   Google Cloud Storage -> Sent directly as input to Vertex AI batch explanation job.
*   BigQuery table -> Sent directly as input to Vertex AI batch explanation job.
*   BigQuery Query -> Not supported.
*   Vertex AI batch explanation job -> Input of batch prediction job is used as input for the Vertex AI batch explanation job.
*   Vertex AI endpoint logging -> Request logging is used as input for Vertex AI batch explanation job.

Check that these datasets meet the requirements for a Vertex AI batch explanation job.

###### Generate model metadata for Vertex Explainable AI
You must specify the explanation specification to use a Vertex AI batch explanation job. Run the following cell to extract metadata from the exported model, which is needed for generating the explanations for a prediction request.

In [ ]:
EXPLANATION_SPEC = ExplanationSpec(
    parameters=ExplanationParameters(
        {"sampled_shapley_attribution": {"path_count": 2}}
    ),
    metadata=ExplanationMetadata(
        inputs={
            "cnt_ad_reward": ExplanationMetadata.InputMetadata(
                {
                    "input_tensor_name": "cnt_ad_reward",
                    "encoding": "IDENTITY",
                    "modality": "numeric",
                }
            ),
            "cnt_challenge_a_friend": ExplanationMetadata.InputMetadata(
                {
                    "input_tensor_name": "cnt_challenge_a_friend",
                    "encoding": "IDENTITY",
                    "modality": "numeric",
                }
            ),
            "cnt_completed_5_levels": ExplanationMetadata.InputMetadata(
                {
                    "input_tensor_name": "cnt_completed_5_levels",
                    "encoding": "IDENTITY",
                    "modality": "numeric",
                }
            ),
            "cnt_level_complete_quickplay": ExplanationMetadata.InputMetadata(
                {
                    "input_tensor_name": "cnt_level_complete_quickplay",
                    "encoding": "IDENTITY",
                    "modality": "numeric",
                }
            ),
            "cnt_level_end_quickplay": ExplanationMetadata.InputMetadata(
                {
                    "input_tensor_name": "cnt_level_end_quickplay",
                    "encoding": "IDENTITY",
                    "modality": "numeric",
                }
            ),
            "cnt_level_reset_quickplay": ExplanationMetadata.InputMetadata(
                {
                    "input_tensor_name": "cnt_level_reset_quickplay",
                    "encoding": "IDENTITY",
                    "modality": "numeric",
                }
            ),
            "cnt_level_start_quickplay": ExplanationMetadata.InputMetadata(
                {
                    "input_tensor_name": "cnt_level_start_quickplay",
                    "encoding": "IDENTITY",
                    "modality": "numeric",
                }
            ),
            "cnt_post_score": ExplanationMetadata.InputMetadata(
                {
                    "input_tensor_name": "cnt_post_score",
                    "encoding": "IDENTITY",
                    "modality": "numeric",
                }
            ),
            "cnt_spend_virtual_currency": ExplanationMetadata.InputMetadata(
                {
                    "input_tensor_name": "cnt_spend_virtual_currency",
                    "encoding": "IDENTITY",
                    "modality": "numeric",
                }
            ),
            "cnt_use_extra_steps": ExplanationMetadata.InputMetadata(
                {
                    "input_tensor_name": "cnt_use_extra_steps",
                    "encoding": "IDENTITY",
                    "modality": "numeric",
                }
            ),
            "cnt_user_engagement": ExplanationMetadata.InputMetadata(
                {
                    "input_tensor_name": "cnt_user_engagement",
                    "encoding": "IDENTITY",
                    "modality": "numeric",
                }
            ),
            "country": ExplanationMetadata.InputMetadata(
                {
                    "input_tensor_name": "country",
                    "encoding": "IDENTITY",
                    "modality": "categorical",
                }
            ),
            "dayofweek": ExplanationMetadata.InputMetadata(
                {
                    "input_tensor_name": "dayofweek",
                    "encoding": "IDENTITY",
                    "modality": "numeric",
                }
            ),
            "julianday": ExplanationMetadata.InputMetadata(
                {
                    "input_tensor_name": "julianday",
                    "encoding": "IDENTITY",
                    "modality": "numeric",
                }
            ),
            "language": ExplanationMetadata.InputMetadata(
                {
                    "input_tensor_name": "language",
                    "encoding": "IDENTITY",
                    "modality": "categorical",
                }
            ),
            "month": ExplanationMetadata.InputMetadata(
                {
                    "input_tensor_name": "month",
                    "encoding": "IDENTITY",
                    "modality": "numeric",
                }
            ),
            "operating_system": ExplanationMetadata.InputMetadata(
                {
                    "input_tensor_name": "operating_system",
                    "encoding": "IDENTITY",
                    "modality": "categorical",
                }
            ),
            "user_pseudo_id": ExplanationMetadata.InputMetadata(
                {
                    "input_tensor_name": "user_pseudo_id",
                    "encoding": "IDENTITY",
                    "modality": "categorical",
                }
            ),
        },
        outputs={
            "churned_probs": ExplanationMetadata.OutputMetadata(
                {"output_tensor_name": "churned_probs"}
            )
        },
    ),
)

In [ ]:
FEATURE_ATTRIBUTION_BASELINE_DATASET = (
    f"{BUCKET_URI}/model-monitoring/churn/churn_no_ground_truth.jsonl"
)
! gcloud storage cp gs://cloud-samples-data/vertex-ai/model-monitoring/churn/churn_no_ground_truth.jsonl $FEATURE_ATTRIBUTION_BASELINE_DATASET

In [ ]:
TIMESTAMP = pd.Timestamp.utcnow().strftime("%Y%m%d%H%M%S")
JOB_DISPLAY_NAME = f"churn_model_monitoring_job_{TIMESTAMP}"
BASELINE_DATASET = ml_monitoring.spec.MonitoringInput(
    gcs_uri=FEATURE_ATTRIBUTION_BASELINE_DATASET, data_format="jsonl"
)
TARGET_DATASET = ml_monitoring.spec.MonitoringInput(endpoints=[endpoint.resource_name])
model_monitoring_job_2 = my_model_monitor.run(
    display_name=JOB_DISPLAY_NAME,
    baseline_dataset=BASELINE_DATASET,
    target_dataset=TARGET_DATASET,
    tabular_objective_spec=ml_monitoring.spec.TabularObjective(
        # Feature attribution spec.
        feature_attribution_spec=FEATURE_ATTRIBUTION_SPEC
    ),
    # You must have a Explanation spec for feature attribution monitoring.
    # You can specify the explanation spec in the Model, Model monitor, or the Model monitoring job.
    explanation_spec=EXPLANATION_SPEC,
    notification_spec=NOTIFICATION_SPEC,
    output_spec=OUTPUT_SPEC,
)

##### List Model Monitoring Jobs

In [ ]:
my_model_monitor.list_jobs()

### Step 6: Wait for the Model Monitoring Job to run and verify the result

#### Verify results through email

Once the model monitoring job begins running, you will receive an email as follows:

<img src="https://services.google.com/fh/files/misc/create_job_email.png" />

After the monitoring job is complete, if any anomalies are detected, you receive an email similar to the following one:

<img src="https://services.google.com/fh/files/misc/job_anomalies_email.png" />

#### Check monitoring metrics: Google Cloud Console

To view Model Monitoring metrics in the [Google Cloud Console](https://console.cloud.google.com/vertex-ai/model-monitoring/model-monitors), go to the **Monitoring** tab under **Vertex AI.**

<img src="https://storage.googleapis.com/cmm-public-data/images/endpoint_jobs_details.gif" />

### Step 7: Schedule Continous Model Monitoring Jobs

To set up continous model monitoring, follow the example below to create a schedule. You can create multiple schedules for your model monitor.


The following example monitors drift in both input features and prediction outputs. The schedule is configured to activate the model monitoring job every hour on the hour, such as at 00:00, 01:00, and so on. Each job analyzes data from the preceding one-hour window. For instance, if a job is scheduled for 6:00 a.m., it analyzes the data collected from 5:00 a.m. to 6:00 a.m.

#### Send more traffic to the endpoint

In [ ]:
bqclient = bigquery.Client(project=PROJECT)
table = bigquery.TableReference.from_string(DATASET_BQ_URI[5:])
rows = bqclient.list_rows(table, max_results=1000)
count = 0
for row in rows:
    instance = {}
    for key, value in row.items():
        if key == GROUND_TRUTH:
            continue
        if value is None:
            value = ""
        instance[key] = value
    endpoint.predict(instances=[instance])
    # Print progress
    if count % 100 == 0:
        print(f"Sent: {count} requests")
    count = count + 1
    time.sleep(0.01)

#### Create a schedule

In [ ]:
# Every 1 hour at :00, for example 1:00, 2:00..
CRON = "0 * * * *"  # @param {type:"string"}

In [ ]:
SCHEDULE_DISPLAY_NAME = "churn-continous-drift-detection"

# Training dataset as baseline
BASELINE_DATASET = TRAINING_DATASET

TARGET_DATASET = ml_monitoring.spec.MonitoringInput(
    endpoints=[endpoint.resource_name],
    window="1h",
)

model_monitoring_schedule = my_model_monitor.create_schedule(
    display_name=SCHEDULE_DISPLAY_NAME,
    cron=CRON,
    baseline_dataset=BASELINE_DATASET,
    target_dataset=TARGET_DATASET,
    tabular_objective_spec=ml_monitoring.spec.TabularObjective(
        # Input feature drift spec.
        feature_drift_spec=FEATURE_DRIFT_SPEC,
        # Prediction output drift spec.
        prediction_output_drift_spec=PREDICTION_OUTPUT_DRIFT_SPEC,
    ),
    notification_spec=NOTIFICATION_SPEC,
    output_spec=OUTPUT_SPEC,
)

SCHEDULE_RESOURCE_NAME = model_monitoring_schedule.name
print(f"Schedule {SCHEDULE_RESOURCE_NAME} created.")

#### Pause Schedule

Run the following to pause the model monitoring schedule:

In [ ]:
my_model_monitor.pause_schedule(schedule_name=SCHEDULE_RESOURCE_NAME)

#### Resume Schedule

Run the following to resume a paused model monitoring schedule:

In [ ]:
my_model_monitor.resume_schedule(schedule_name=SCHEDULE_RESOURCE_NAME)

#### Update Schedule

Run the following to update the model monitoring schedule:

In [ ]:
# Update to run every 1 hour at :30, for example 0:30, 1:30, 2:00..
my_model_monitor.update_schedule(
    schedule_name=SCHEDULE_RESOURCE_NAME, cron="30 * * * *"
)

#### Check monitoring schedules in Google Cloud Console

To check your Model Monitoring schedule in the Google Cloud Console, go to the Monitor tab under Vertex AI.


<img src="https://services.google.com/fh/files/misc/endpoint_schedules.gif" />

### Step 8: Clean Up

If you no longer need your model monitoring resources: uncomment the next cell and run the following to clean resources

In [ ]:
# # When no jobs are running, delete the schedule and all the jobs.
# my_model_monitor.delete(force=True)

# # Undeploy endpoint
# endpoint.undeploy_all()
# endpoint.delete()

# # Delete the model
# model.delete()